# 00 · Data manifest and conversion to analysis-ready tables

Objective O1: acquire, validate and integrate the datasets into analysis-ready Parquet tables.

1. Check every raw file against `data/raw/manifest.csv` (size, and optionally SHA-256).
2. Clean the collision records (D1) and build the composite city-hour weather table (D5).
3. Convert the 2023–2025 WYS speed records (D3) to Parquet partitioned by year/month. All later steps, including E4, read this Parquet copy.
4. Save the WYS sign locations.

This notebook only prepares data. It is not part of any timed experiment.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

## 1. Manifest check

In [ ]:
import hashlib
raw_dir = path(cfg, "raw")
man = pd.read_csv(raw_dir / "manifest.csv")
man["exists"] = man["file"].map(lambda f: (raw_dir / f).exists())
man["size_ok"] = [(raw_dir / f).exists() and (raw_dir / f).stat().st_size == b for f, b in zip(man["file"], man["bytes"])]
print(f"{len(man)} files in manifest | present: {man['exists'].sum()} | size matches: {man['size_ok'].sum()}")
on_disk = {str(p.relative_to(raw_dir)) for p in raw_dir.rglob("*") if p.is_file()
           and p.name not in (".DS_Store", "manifest.csv", "download.log") and "backup" not in p.name}
print("on disk but not in manifest:", sorted(on_disk - set(man["file"])))
man.loc[~man["size_ok"], ["file", "bytes"]]

In [ ]:
# Optional: full SHA-256 check (reads all 17 GB, about 1-2 minutes per 10 GB)
VERIFY_SHA = False
if VERIFY_SHA:
    def sha256(p, buf=1 << 24):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            while chunk := f.read(buf):
                h.update(chunk)
        return h.hexdigest()
    man["sha_ok"] = [sha256(raw_dir / f) == s if e else False for f, s, e in zip(man["file"], man["sha256"], man["exists"])]
    print("SHA-256 matches:", man["sha_ok"].sum(), "/", len(man))

## 2. Collisions (D1)

In [ ]:
from src import collisions as C
raw_coll = C.load_raw_collisions(cfg)
coll = C.clean_collisions(raw_coll)
coll.to_parquet(PROC / "collisions.parquet", index=False)
C.quality_summary(raw_coll, coll)

## 3. Composite weather table (D5) with calendar features

In [ ]:
from src import weather as W
display(W.coverage_table(cfg))      # share of hours with data, by station and variable
wx = W.add_calendar(W.build_composite_weather(cfg), cfg)
wx.to_parquet(PROC / "weather_hourly.parquet", index=False)
print(len(wx), "city-hours")
wx[["rain", "snow", "vis_lt1", "vis_1to5", "dst_transition", "precip_filled", "precip_from_daily", "temp_c_filled"]].mean().round(4)

## 4. WYS speed records (D3) → Parquet

In [ ]:
from src import wys
spark = get_spark(cfg)
spark.sparkContext.setLogLevel("ERROR")
pq_path = wys.convert_to_parquet(spark, cfg)   # 2023-2025, about 10 minutes
wys_raw = spark.read.parquet(pq_path)
by_year = wys_raw.groupBy("year").count().orderBy("year").toPandas()
by_year.loc["total"] = ["", by_year["count"].sum()]
by_year

In [ ]:
locations = wys.load_locations(cfg)
locations.to_parquet(PROC / "wys_locations.parquet", index=False)
print(len(locations), "signs;", locations["speed_limit"].value_counts().to_dict())

In [ ]:
spark.stop()